## Exercise 3-1: Building the Option Volatility Smirk from OptionMetrics

In this exercise we build a **firm-year panel of the option-implied volatility smirk** from scratch, using the WRDS querying skills from Module 3-1.

**Background.** Equity options on the same stock and with the same maturity do not trade at the same implied volatility (IV). Plotting IV against moneyness for individual US stocks produces a downward-sloping curve — a *smirk*: deep out-of-the-money (OTM) puts trade at systematically higher implied volatilities than at-the-money (ATM) calls. [Xing, Zhang and Zhao (2010, *JFQA*)](https://doi.org/10.1017/S0022109010000463) argue that the steepness of this smirk reveals what informed investors expect about the *left tail* of the return distribution: when demand for crash protection is high, OTM puts become expensive relative to ATM calls and the smirk steepens. They define

$$\text{SMIRK}_{i,t} \;=\; IV^{\text{OTM put}}_{i,t} \;-\; IV^{\text{ATM call}}_{i,t}$$

and show it predicts future stock returns and jump risk. [Kim, Li, Lu and Yu (2016, *JAE*)](https://doi.org/10.1016/j.jacceco.2015.12.003) use this daily measure, averaged to the firm-year, as a forward-looking, market-based proxy for **expected crash risk**.

The pipeline has six stages:

1. For every stock (`secid`) and every trading day, compute the **open-interest-weighted average IV** of ATM calls and of OTM puts, from OptionMetrics' daily option price file.
2. Take the difference to get the **daily smirk**.
3. Map OptionMetrics `secid` → CUSIP → CRSP `permno`.
4. Pull **Compustat** fundamentals and build the control variables.
5. Link Compustat `gvkey` → `permno` through the **CCM** link history.
6. Average the daily smirk within each firm-year window to get the final panel.

## Step 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import wrds

## Step 1. Explore the OptionMetrics library

In [ ]:
db = wrds.Connection(wrds_username='leonardl')

In [ ]:
assert "optionm" in db.list_libraries(), "OptionMetrics library not in your WRDS subscription."

In [ ]:
# The daily option price files, one table per calendar year
print([int(t[-4:]) for t in db.list_tables(library="optionm") if "opprcd" in t])

In [ ]:
db.describe_table(library="optionm", table="opprcd2022")

In [ ]:
db.close()

## Step 2. Daily implied volatility of ATM calls and OTM puts

A *single* year of this table holds tens of millions of option-day observations. Downloading it and aggregating in pandas would be slow and would exhaust your laptop's memory. Thus, we use `GROUP BY` on the WRDS server, so only one row per stock-day-type comes back.

Refer to Appendix B of Kim et al. (2016) for the detailed definition of `IV_Smirk`.

In [ ]:
def get_iv(start_year: int, end_year: int) -> pd.DataFrame:
    """
    Get the implied volatility for all options in the OptionMetrics database between start_year and end_year.
    """

    df_out = pd.DataFrame()
    with wrds.Connection(wrds_username='leonardl') as db:
        for year in range(start_year, end_year + 1):
            iv_query = f"""
                    SELECT secid,
                        date,
                        cp_flag,
                        SUM(impl_volatility * open_interest) / SUM(open_interest) AS iv
                    FROM optionm.opprcd{year}
                    WHERE impl_volatility > 0.03
                    AND impl_volatility < 2
                    AND open_interest > 0
                    AND volume IS NOT NULL
                    AND best_bid IS NOT NULL
                    AND best_offer IS NOT NULL
                    AND best_offer >= best_bid
                    AND ( (delta >  0.375 AND delta <  0.625)
                        OR (delta > -0.375 AND delta < -0.125) )
                    GROUP BY secid, date, cp_flag
                """
            _temp_df = db.raw_sql(iv_query, date_cols=["date"])
            _temp_df["secid"] = _temp_df["secid"].astype("int32") # save memory
            _temp_df["iv"] = _temp_df["iv"].astype("float32") # save memory
            df_out = pd.concat([df_out, _temp_df], ignore_index=True)
    
    return df_out

In [ ]:
iv_sample = get_iv(start_year=2018, end_year=2019)

In [ ]:
iv_sample.describe()

## Step 3. The daily volatility smirk

In [ ]:
iv_daily = (
    iv_sample
    .pivot(index=["secid", "date"], columns="cp_flag", values="iv")
    .rename(columns={"C": "atm_iv", "P": "otm_put_iv"})
    .reset_index()
    .dropna()
)

iv_daily["iv_skew"] = iv_daily["otm_put_iv"] - iv_daily["atm_iv"]
iv_daily.head()

In [ ]:
iv_daily[["atm_iv", "otm_put_iv", "iv_skew"]].describe([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

## Step 4. Link `secid` to `permno`

In [ ]:
with wrds.Connection(wrds_username='leonardl') as db:
    # OptionMetrics security file: one row per secid, carrying its CUSIP
    secid_cusip = db.raw_sql("SELECT secid, cusip FROM optionm.securd")
    secid_cusip["secid"] = secid_cusip["secid"].astype("int32")
    
    # CRSP name history
    stocknames = db.raw_sql(
        """
        SELECT permno, cusip, namedt, nameenddt
        FROM crsp.stocknames_v2
        WHERE cusip IS NOT NULL
        """,
        date_cols=["namedt", "nameenddt"],
    )

In [ ]:
secid_cusip.head()

In [ ]:
stocknames.head()

In [ ]:
daily_permno = (
    iv_daily
    .merge(secid_cusip, on="secid", how="inner")
    .merge(stocknames, on="cusip", how="inner")
    .query("(namedt <= date) & (date <= nameenddt | nameenddt.isna())") # Keep only matches where the option date falls inside the CUSIP's validity window.
)
daily_permno.head()

In [ ]:
daily_permno["permno"] = daily_permno["permno"].astype("int32")
daily_permno = daily_permno.sort_values(["permno", "date"]).reset_index(drop=True)

print(f"{len(daily_permno):,} firm-day observations, of {daily_permno['permno'].nunique():,} unique permnos")

In [ ]:
# Diagnostic: a CUSIP that maps to two permnos over overlapping windows would produce duplicate firm-days and double-count in the averages below.
assert daily_permno.duplicated(subset=["permno", "date"]).sum() == 0, "Duplicate permno-date rows found!"

## Step 5. Compustat fundamentals and the CCM link

In [ ]:
start_year = 2018
end_year = 2018
query = f"""
        SELECT compustat.*, ccm.permno 
		FROM 
			(
            SELECT gvkey, datadate, fyear, at, ceq, sale, ni,
                csho * prcc_f AS mcap, 
                dltt / at AS Leverage
				FROM comp.funda 
				WHERE fyear BETWEEN {start_year} AND {end_year}
                	AND fyear IS NOT NULL 
					AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C' 
					AND at IS NOT NULL 
                    AND at > 0 
            ) AS compustat, 
            (
            SELECT gvkey, lpermno AS permno, linkdt, linkenddt
				FROM crsp.ccmxpf_linktable
				WHERE linktype in ('LU','LC','LS')
            ) AS ccm
		WHERE compustat.gvkey = ccm.gvkey
			AND (compustat.datadate >= ccm.linkdt OR ccm.linkdt IS NULL)
			AND (compustat.datadate <= ccm.linkenddt OR ccm.linkenddt IS NULL)
		
        ORDER BY compustat.gvkey, fyear, datadate DESC, permno
        """
with wrds.Connection(wrds_username='leonardl') as db:
    CCM = (
        db.raw_sql(query, date_cols=['datadate'])
        .drop_duplicates(subset=['gvkey', 'fyear'])
        )
CCM["Size"] = np.log(CCM['mcap'].where(CCM['mcap'] > 0))
CCM["MB"] = CCM['mcap'] / CCM["ceq"].where(CCM["ceq"] != 0)
CCM[["Size", "leverage", "MB"]].describe()
CCM.head()

## Step 6. Collapsing the daily smirk into a firm-year measure

In [ ]:
smirk = (
    pd.merge(
        CCM,
        daily_permno[["permno", "date", "atm_iv", "iv_skew"]],
        on = ["permno"],
        how = "inner"
    )
    .loc[lambda d: (
        (d["datadate"] - pd.DateOffset(months=8) < d["date"]) &
        (d["date"] <= d["datadate"] + pd.DateOffset(months=3))
    )]
    .sort_values(by=["gvkey", "fyear", "date"])
)

In [ ]:
smirk_yearly = (
    smirk
    .groupby(["gvkey", "fyear"])
    .agg(
        Size = ("Size", "mean"),
        Leverage = ("leverage", "mean"),
        MB = ("MB", "mean"),
        atm_iv = ("atm_iv", "mean"),
        iv_skew = ("iv_skew", "mean"),
        n_obs = ("iv_skew", "count")
    )
    .reset_index()
)

## Step 7. Check the result

In [ ]:
smirk_yearly[["iv_skew", "atm_iv", "n_obs", "Size", "Leverage", "MB"]].describe([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.kdeplot(data = smirk_yearly[smirk_yearly['n_obs'] >= 200], 
            x = 'iv_skew', 
            color="steelblue", 
            fill = True, 
            linewidth=2)

plt.title("Distribution of Average Option Volatility Smirk", fontsize=14)
plt.xlabel("Average option volatility smirk")
plt.ylabel("Density")
plt.grid(axis="y", linewidth=0.5)

plt.show()